# **Building a Decoder-Only Transformer from Scratch**

Note:
1. You can use Google Colab to directly work on this notebook.
2. This notebook takes inspiration from Andrej Karpathy's excellent video [“Let's build GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY&t=4297s).” If you'd like a deeper understanding of the concepts, you can check out the original video.

---

Transformers have revolutionized natural language processing (NLP), powering modern large language models (LLMs) like GPT-4 and Gemini.
At their core, Transformers are neural architectures built around the key idea of attention, which allows each token in a sequence to attend to every other token, capturing long-range dependencies far more effectively than architectures like RNNs or LSTMs.

In this tutorial, we will build a simplified, **character-level decoder-only Transformer model** from scratch in PyTorch, and train it on a small dataset - The Adventures of Sherlock Holmes by Arthur Conan Doyle.


**What Is a Decoder-Only Transformer?**

A Transformer can be broadly divided into two parts:

Encoder: Reads and encodes the entire input sequence into a contextual representation.

Decoder: Generates an output sequence, attending to both previously generated tokens (via causal self-attention) and optionally to encoder outputs.

In a decoder-only Transformer (like GPT), there is no encoder.
Instead, the model learns to predict the next token given all previous tokens in a sequence.
This setup makes it ideal for language modeling and text generation tasks.

The process is autoregressive i.e., the model generates text one token at a time, feeding each new token back as input.

----

**Notebook Structure**

Here's how we'll build our model step by step:

**1. Data Preparation**

Download the Sherlock Holmes text from Project Gutenberg.

Clean the text to extract the main content.


**2. Tokenization**

Build a character-level vocabulary — each unique character becomes a token.

Create encoder and decoder functions to map between text and integer IDs. This is to encode and decode vocabulary, and different from the encoder-decoder of a transformer.

**3. Data preparation**

Divide the dataset into training and validation subsets.

**4. Data Loader**

Implement a custom batching function that samples random text chunks (context windows) for training.

Each batch provides input sequences x and corresponding next-character targets y.

**5. Model Architecture**

We’ll progressively assemble the Transformer model piece by piece:

Single Attention Head → scaled dot-product attention with causal masking.

Multi-Head Attention → multiple attention heads in parallel.

FeedForward Network (FFN) → nonlinear token-wise transformation.

Transformer Block → combines attention + feedforward + layer normalization + residual connections.

Full Model (LanguageModel) → token embeddings, positional embeddings, stacked Transformer blocks, and an output projection head.

**6. Training Loop**

Compute cross-entropy loss between predicted and true next tokens.

Train with AdamW optimizer, periodically evaluating on validation data.

**7. Text Generation**

After training, use the model to generate new text autoregressively.

----

The goal of this notebook is not to train a large or powerful model, but to understand the inner workings of a Transformer by building each component manually and observing how it learns. We'll implement a small, character-level decoder-only Transformer trained on a single novel. Despite using a tiny dataset, fewer parameters, a short context window, and character-level tokenization, the model will still demonstrate meaningful learning: generated text will gradually evolve from random noise to somewhat English-like patterns.

**0. Import Libraries and defining hyperparameters**

Import PyTorch libraries and set seed for reproducibility

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(67458)

# hyperparameters

batch_size = 16   # number of sequences processed in one training step
context_length = 32   # length of each input sequence

max_iters =  5000   # total number of training iterations
eval_interval = 100   # evaluate model on train/val every N iterations
eval_iters = 200    # number of batches used to estimate average eval loss

learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n_embd = 64   # embedding dimension — size of token and position vectors
n_head = 4    # number of attention heads per Transformer block
n_layer = 4    # number of Transformer blocks (depth of the model)
dropout = 0.2    # dropout probability for regularization


**1. Data Preparation**

Download, read, and clean the data file.

In [6]:
!wget https://www.gutenberg.org/files/1661/1661-0.txt -O sherlock.txt


--2025-10-15 14:32:26--  https://www.gutenberg.org/files/1661/1661-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607504 (593K) [text/plain]
Saving to: ‘sherlock.txt’

sherlock.txt        100%[===================>] 593.27K  --.-KB/s    in 0.06s   

2025-10-15 14:32:26 (9.75 MB/s) - ‘sherlock.txt’ saved [607504/607504]



In [7]:
with open("sherlock.txt") as f:
    text = f.read()
start = text.find("I. A SCANDAL IN BOHEMIA")
end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***")
text = text[start:end].strip()
open("sherlock_clean.txt", "w").write(text)
text[:1000]

'I. A SCANDAL IN BOHEMIA\n\n\nI.\n\nTo Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained\nreasoner to admit such intrusions into his own delicate and finely\nadjusted temperament was to introduce a distracting factor which might\nthrow a doubt upon all his mental results. Grit in a sensitive\ninstrument, or a crack in one of his own high-power lens

**2. Tokenization**

Before a model can learn language, we must convert raw text into tokens (the discrete symbols it will operate on).
Each token is assigned a numeric ID, which is then mapped to a continuous embedding vector during training.

There are several common strategies for tokenization, differing mainly in the size and granularity of the vocabulary: Character-level, Subword-level, Byte-Pair Encoding (BPE), and more.

In this notebook, we use character-level tokenization — every individual character (letters, digits, punctuation, whitespace) becomes a token.
This approach keeps the implementation simple and transparent, so we can focus on understanding how the Transformer learns, not on preprocessing complexity.

However, character-level models must process much longer sequences (e.g., the word “Sherlock” becomes 8 tokens instead of 1), which makes learning slower and less efficient.
For that reason, real-world large language models use subword tokenization (e.g., BPE) to represent text more compactly and efficiently, while still being able to compose unseen words.

In [33]:
# Before we can feed text into our model, we need to define its vocabulary —
# the complete set of unique characters that appear in the dataset and it will be the universe of symbols the model can understand.
#
# 1. Extract all the unique characters from the variable `text`.
# 2. Sort them to ensure a consistent ordering.
# 3. Store the result in a list called `vocabulary`.
# 4. Compute the total number of unique characters and store it in `vocabulary_size`.
# 5. Finally, print both the vocabulary size and the list of unique characters.


vocabulary = ...
vocabulary_size = ...

print(f"Unique characters: {vocabulary}")
print(f"Vocabulary size: {vocabulary_size}")

Unique characters: Ellipsis
Vocabulary size: Ellipsis


Neural networks work with numbers, not text.
To train our Transformer, we need a way to represent each character as a unique integer ID.

This process of mapping symbols to integers is called encoding.
It allows us to convert text into sequences of numbers that the model can process, and later convert model outputs (integers) back into readable text.

We’ll create two mappings:

str_to_int: from each character to its corresponding integer ID.

int_to_str: from each integer ID back to its original character.

These will form the foundation of our encode and decode functions later.

In [34]:
# Now that we have our vocabulary, we need to map each character to a unique integer ID.
#
# 1. Index each character in `vocabulary`
# 2. Store these mappings in two dictionaries:
#    - `str_to_int`: maps each character (string) → integer ID
#    - `int_to_str`: maps each integer ID → character (string)
# 3. Print both dictionaries to verify that the mappings are correct.
#
# Example (simplified):
# str_to_int = {'a': 0, 'b': 1, 'c': 2}
# int_to_str = {0: 'a', 1: 'b', 2: 'c'}

str_to_int = ...
int_to_str = ...

print("Mappings:")
print(str_to_int)
print(int_to_str)


Mappings:
Ellipsis
Ellipsis


Now that we can map each character to a unique integer ID, we need simple helper functions to switch between these two representations whenever needed.

Encoding converts text (strings) into numerical form (lists of integers) so the model can process it.

Decoding converts the model’s numerical predictions back into readable text.

These two operations form the essential bridge between human-readable text and machine-understandable numbers.
Later, we’ll use them to convert our dataset into tensors and to interpret the model’s generated output as text again.


In [35]:
# 1. Create a function called `encode` that:
#    - Takes a string as input.
#    - Returns a list of integers corresponding to each character in the string.
#
# 2. Create a function called `decode` that:
#    - Takes a list of integers as input.
#    - Returns the reconstructed string by mapping each integer back to its character.
#
# Example:
# text = "hello"
# encoded = encode(text)      # [3, 2, 4, 4, 5]
# decoded = decode(encoded)   # "hello"
#
# Sanity check:
# The decoded text should match the original input text.


encode = ...
decode = ...


**3: Preparing Training and Validation Data**

Before we train our model, we need to convert our entire text into a numerical format and split it into training and validation sets.

The training set is used to learn model parameters, while the validation set is used to evaluate model performance on unseen data.

In [36]:
# 1. Use the `encode` function to convert the full text into a list of integers.
# 2. Convert this list into a PyTorch tensor called `data` (use dtype=torch.long).
# 3. Split `data` into:
#    - `train_data`: the first 90% of the data
#    - `val_data`: the remaining 10%
# 4. Print the lengths of both to verify the split.


data = ...

train_data = ...
val_data = ...


**4. Data Loader**

Transformers can’t train on the entire dataset at once, so we sample random chunks of fixed length called a context window.

Within a single context window, there are multiple training pairs, for example in one example of context window of length 5 [43, 56, 67, 89, 90] there are 4 training examples [43] -> [56], [43, 56] -> [67], [43, 56, 67] -> [89], [43, 56, 67, 89] -> [90].

We also batch multiple windows for efficiency.




In [38]:
# Write data_loader function that returns:
# x: tensor of train sequences (batch_size, context_length)
# y: tensor of next-token labels (batch_size, context_length), which is x shifted by one position because we are working with next word prediction.

# 1) Choose which dataset to sample from — use the training set for 'train' and the validation set for 'val'.
#
# 2) Randomly select several valid starting positions in the data, based on the desired number of examples in the batch.
#    Make sure each starting point allows a full context window to fit inside the data (for train and val).
#
# 3) For each selected start position, extract a small slice of tokens of length equal to the context length — this will be your train sequence (x).
#
# 4) For each train sequence, create a corresponding target sequence (y) by shifting it one step forward in time — i.e., each element in y is the
#    "next token" that follows the corresponding element in x.
#
# 5) Move both to the correct device (`device`) and return them.
#
# Toy Example:
#   data = tensor([10,11,12,13,14,15,16,17])  # len = 8
#   context_length = 4, batch_size = 2
#   Valid start indices: 0..(len(data) - context_length - 1) = 3  → {0,1,2,3}
#   Suppose ix = [1, 3]
#     x = [[11,12,13,14],[13,14,15,16]]
#     y = [[12,13,14,15],[14,15,16,17]]


def data_loader(split):
  """
  Generates a small batch of train (x) and target (y) sequences
  for training or validation.

  Parameters
  ----------
  split : str
      Which dataset split to sample from.
      - 'train' → use training data
      - 'val'   → use validation data

  Returns
  -------
  x : torch.Tensor
      Train batch of shape (batch_size, context_length),
      containing integer token IDs.

  y : torch.Tensor
      Target batch of shape (batch_size, context_length),
      where each sequence is `x` shifted one token to the right.
  """
  # Your implementation here
  return None, None



**5. Model Architecture**

**(a). Self Attention**

Attention captures long-range dependencies by allowing each token to attend to other tokens in the sequence when forming its representation.
In essence, attention computes a weighted combination of all tokens, where the weights represent how relevant each token is to the current one.

---

Types of Attention

Self-Attention:
Each token attends to other tokens in the same sequence.
This helps the model understand relationships within a single text (e.g., subject–verb dependencies, context around words).

Cross-Attention:
Tokens in one sequence attend to tokens in another sequence.
For example, in machine translation, the decoder attends to the encoder outputs to generate translated text.

---

In a decoder-only Transformer, we only use self-attention.
Each token builds its contextual representation based on previous and current tokens.
Because our model is autoregressive (predicting one token at a time), we apply causal masking to prevent the model from “peeking” at future tokens during training.

In [39]:
# Implement a Single Self-Attention Head
#
# 1. Create three linear layers (no bias):
#    - `key`: projects the input to key vectors.
#    - `query`: projects the input to query vectors.
#    - `value`: projects the input to value vectors.
#
# 2. In the forward pass:
#    - Compute Q (queries), K (keys), and V (values) from the input `x`.
#    - For efficiency, use `F.scaled_dot_product_attention(q, k, v, is_causal=True)`
#      to perform **causal self-attention** — ensuring each position only
#      attends to the past and present tokens, never the future. (Bonus point if you write your own code for self-attention calculation)
#    - Optional: apply dropout
#
# 3. Return the resulting context-aware output.


class Head(nn.Module):
    """
    A single head of self-attention

    Parameters
    ----------
    head_size : int
        The dimensionality of the key, query, and value projections
        for this attention head.

    Attributes
    ----------
    key : nn.Linear
        Projects input embeddings into key vectors.
    query : nn.Linear
        Projects input embeddings into query vectors.
    value : nn.Linear
        Projects input embeddings into value vectors.

    Methods
    -------
    forward(x : torch.Tensor) -> torch.Tensor
        Computes causal self-attention:
        - Projects the input into Q, K, and V subspaces.
        - Applies scaled dot-product attention with causal masking.
        - Returns the resulting context-aware representations of shape
          (batch_size, context_length, head_size).
    """


    def __init__(self, head_size):
        super().__init__()
        self.key = ...
        self.query = ...
        self.value = ...

    def forward(self, x):
        q = ...
        k = ...
        v = ...
        out = ...
        return out

(b) Multi-Head Self-Attention

A single attention head tends to specialize in one kind of relationship (e.g., local proximity, syntax, or long-range links).
Multi-head attention runs several heads in parallel on the same input so that each head can focus on a different subspace of information.
Each head projects from the full embedding dimension n_embd down to a smaller head_size; after computing attention independently, we concatenate all head outputs and apply a final linear projection to mix them:

n_embd = num_heads * head_size (concatenation restores the full dimensionality)

This design increases the capacity of the model to capture richer, complementary patterns in the sequence.

In [40]:
# Implement Multi-Head Self-Attention**
#
# Goal:
# Combine multiple self-attention heads that run in parallel, then project
# their concatenated outputs back to the model dimension.
#
# 1) Instantiate `num_heads` independent `Head` modules, each with `head_size`.
#    - Store them in `nn.ModuleList` so their parameters are tracked by PyTorch.
#
# 2) In the forward pass:
#    - For each head, compute its output on the same input `x`.
#    - Concatenate the head outputs along the feature dimension.
#    - Apply a final linear projection.
#    - (Optional) Apply dropout after the projection for regularization.


class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention

    Parameters
    ----------
    num_heads : int
        Number of parallel attention heads.
    head_size : int
        Dimensionality of each head's key/query/value space. Typically,
        `head_size = n_embd // num_heads`.

    Attributes
    ----------
    heads : nn.ModuleList
        A list of `Head` modules, each with its own Q/K/V projections.
    proj : nn.Linear
        Final linear layer mapping concatenated heads back to `n_embd`.
        (Optional dropout may follow)

    Methods
    -------
    forward(x : torch.Tensor) -> torch.Tensor
        Computes all heads on input `x`, concatenates along the feature dim,
        applies the output projection, and returns a tensor of shape
        (batch_size, context_length, n_embd).
    """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = ...
        self.proj = ...

    def forward(self, x):
        out = ...
        return out

**(c) FeedForward Network**

In the Transformer architecture, each block has two key components:

Self-Attention: enables communication between tokens — each token looks at others and gathers contextual information.

FeedForward Network (FFN): performs computation on each token independently, enriching its representation after attention has shared information.

Once the model has integrated context through attention, the FFN acts like a small multilayer perceptron (MLP) that transforms each token’s embedding via nonlinear transformations.
It applies two linear layers with a ReLU activation (or sometimes GELU) in between.
The first layer expands the embedding dimension by a factor of 4, a design choice from “Attention Is All You Need” (Vaswani et al., 2017) that increases the model’s expressive power, before projecting it back down.
Finally, dropout is added for regularization.

Mathematically:

$$
\begin{aligned}
\mathrm{FFN}(x) &= \max(0,\, xW_1 + b_1)\, W_2 + b_2 \\
\end{aligned}
$$
where,
$$
\begin{aligned}
W_1 &\in \mathbb{R}^{n_{\text{embd}} \times 4n_{\text{embd}}}, \quad b_1 \in \mathbb{R}^{4n_{\text{embd}}} \\
W_2 &\in \mathbb{R}^{4n_{\text{embd}} \times n_{\text{embd}}}, \quad b_2 \in \mathbb{R}^{n_{\text{embd}}}
\end{aligned}
$$


In [41]:
# Implement the FeedForward Network (FFN)
#
# 1) Build a small two-layer MLP inside the class:
#    - First linear layer expands dimensionality from `n_embd` → `4 * n_embd`
#    - Apply a ReLU non-linearity
#    - Second linear layer projects back from `4 * n_embd` → `n_embd`
#    - Add dropout for regularization
#
# 2) Wrap the layers in an `nn.Sequential` container for clean organization.


class FeedFoward(nn.Module):
    """
    The FeedForward Network (FFN)

    Parameters
    ----------
    n_embd : int
        Dimensionality of the token embeddings (input and output size).

    Attributes
    ----------
    net : nn.Sequential
        Sequential container holding:
        - Linear layer (n_embd → 4 * n_embd)
        - ReLU nonlinearity
        - Linear layer (4 * n_embd → n_embd)
        - Dropout for regularization

    Methods
    -------
    forward(x : torch.Tensor) -> torch.Tensor
        Applies the feedforward transformation to each token independently,
        returning a tensor of the same shape as input
        (batch_size, context_length, n_embd).
    """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            ...
        )

    def forward(self, x):
        out = ...
        return out

**(d) Transformer Block**

So far, we’ve built two fundamental components of the Transformer:

Multi-Head Self-Attention (MHA): enables communication between tokens — each token learns from others in the sequence.

FeedForward Network (FFN): performs computation on each token independently — once tokens have gathered context through attention, the FFN refines and transforms their representations.

The Transformer Block combines these two stages — “talking” (via attention) and “thinking” (via feedforward).
By stacking multiple blocks, the model progressively builds deeper, more abstract representations of the input — similar to how understanding deepens through successive layers of reasoning.

**Residual connections**

Each sublayer (attention and feedforward) is wrapped with a residual connection.

Why it matters:

* It preserves the original information flow across layers.

* It helps prevent vanishing gradients, making training deep networks feasible.

* It stabilizes optimization


**Layer Normalization**

Before each sublayer, we apply Layer Normalization — normalizing activations across the embedding dimension for every token.
This keeps activations well-scaled, balances learning between sublayers, and improves gradient flow.

**This implementation uses the Pre-LN configuration:**

Normalize first → Apply the sublayer → Add the residual.

This differs from the Post-LN design in the original Transformer (where normalization followed residuals).
Pre-LN improves stability in deep models and simplifies optimization.

In [42]:
# Implement the Transformer Block
#
# 1) Initialize the following components:
#    - A MultiHeadAttention layer (defined earlier)
#    - A FeedForward layer (defined earlier)
#    - Two LayerNorm layers (nn.LayerNorm)
#
# 2) In the forward pass:
#    - Apply Pre-LN pattern:
#         a) Normalize `x` → apply multi head attention → add residual connection
#         b) Normalize `x` → apply feedforward → add residual connection
#
# 3) Return the final output.

class Block(nn.Module):
    """
    Parameters
    ----------
    n_embd : int
        Embedding dimension of the input and output.
    n_head : int
        Number of attention heads to use in the multi-head attention layer.

    Attributes
    ----------
    mha : MultiHeadAttention
        Multi-head self-attention layer.
    ffwd : FeedFoward
        Feedforward network.
    ln1, ln2 : nn.LayerNorm
        Layer normalization applied before each sublayer.

    Methods
    -------
    forward(x : torch.Tensor) -> torch.Tensor
        Applies the Transformer block:
            - Normalizes input and applies self-attention with residual.
            - Normalizes result and applies feedforward with residual.
        Returns a refined representation tensor of shape
        (batch_size, context_length, n_embd).
    """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = ...
        self.mha = ...
        self.ffwd = ...
        self.ln1 = ...
        self.ln2 = ...

    def forward(self, x):
        out = ...
        return out

**(e) Full Language Model**

To complete our decoder-only Transformer, we do the following steps:

---

**Token and Position Embeddings**

Token Embeddings:
Convert discrete token IDs into continuous vectors of size n_embd.
This allows the model to learn semantic relationships between tokens.

Position Embeddings:
Encode the position of each token within a sequence.
Since self-attention is inherently order-agnostic, positional information ensures the model understands word order — distinguishing, for instance, “dog bites man” from “man bites dog.”

Together, these embeddings provide both content and positional context for every token in the input.

---

**Stacked Transformer Blocks**

A series of identical Transformer blocks forms the computational core of the model.
Each block performs:

Communication: via multi-head self-attention, where tokens exchange information.

Computation: via a feedforward network that processes each token independently.

Layer normalization and residual connections within each block stabilize training and preserve information flow.
By stacking multiple blocks, the model progressively builds richer hierarchical representations — from local dependencies to broader semantic structures.

---

**Final LayerNorm**

A final normalization step is applied before producing logits.
This ensures stable activation magnitudes and smoother optimization before the output layer.

---

**Output Head**

The output linear layer maps the hidden representations of each token to a probability distribution over the vocabulary:

Each position now predicts the likelihood of every possible next token.

---

**Training Loss**

During training, the model learns to predict the next token given all previous ones.
We use cross-entropy loss between predicted logits and true targets.

Hint:
Before computing the loss, both tensors can be reshaped to:

logits → (batch_size*context_length, vocabulary_size)

targets → (batch_size*context_length,)

This aligns them with PyTorch’s F.cross_entropy expectation.


---

**Generation (Inference)**

At inference time, the model generates text autoregressively — one token at a time:

Feed in the current sequence (cropped to the last context_length tokens).

Compute the next-token probabilities from the final logits.

Sample or choose the most likely token.

Append it to the sequence and repeat.

Over time, the model learns to produce increasingly coherent and language-like text.

In [ ]:
# Build the Full Decoder-Only Language Model
#
# Goal:
# Combine embeddings, stacked Transformer blocks, a final LayerNorm,
# and an output linear head into a complete next-token predictor.
#
# 1) Embeddings: (nn.Embedding can be used)
#    - Create `token_embedding_table`: maps token IDs → embedding vectors (n_embd).
#    - Create `position_embedding_table`: maps positions → embedding vectors (n_embd).
#
# 2) Core model:
#    - Stack `n_layer` Transformer `Block`s in an `nn.Sequential`.
#    - Add a final LayerNorm before the output projection.
#    - Add linear layer to produce logits.
#
# 3) Forward pass:
#    - Inputs: `idx` token IDs; optional `targets`.
#    - Look up token and positional embeddings and sum them.
#    - Run through the stacked blocks and final LayerNorm.
#    - Project to logits.
#    - If `targets` is provided, compute cross-entropy loss
#    - Return `(logits, loss)`.
#
# 4) Generation:
#    - Implement `generate(idx, max_new_tokens)`:
#        * Run forward; take the last time-step logits.
#        * Convert to probabilities (e.g., softmax), sample next token ID.
#        * Append to `idx` and continue.
#    - Return the extended sequence.



class LanguageModel(nn.Module):
    """
    A compact decoder-only Transformer language model.

    Components
    ----------
    - Token embeddings
    - Positional embeddings
    - Stacked Transformer blocks
    - Final LayerNorm
    - Output head

    Forward Pass
    ------------
    Input:
        idx : LongTensor of shape (batch_size, context_length)
            Batch of token IDs.
        targets : Optional[LongTensor] of shape (batch_size, context_length)
            Next-token labels for loss computation.

    Steps:
        1) Embed tokens and positions by sum
        2) Apply stacked Transformer blocks
        3) Final LayerNorm
        4) Linear projection to logits
        5) If `targets` provided compute cross-entropy.

    Returns:
        logits : FloatTensor of shape (batch_size, context_length, vocabulary_size)
            Unnormalized scores for each token in the vocabulary at each position.
        loss : Optional[torch.Tensor]
            Cross-entropy loss if `targets` is provided; otherwise `None`.
    """
    def __init__(self):
        super().__init__()

        self.token_embedding_table = ...
        self.position_embedding_table = ...
        self.blocks = nn.Sequential(...) # add here n_layer number of blocks
        self.ln_f = ... # final layer norm
        self.lm_head = ...

    def forward(self, idx, targets=None):

        tok_emb = ...
        pos_emb = ...
        # next: combine embeddings and pass through blocks, layer norm, and last linear layer to output logits
        logits = ...

        if targets is None:
            loss = None
        else:
            # add reshaping for logits and targets here
            loss = ... # use F.cross_entropy

        return logits, loss

    def generate(self, idx, max_new_tokens): # no need to change this function: it can be directly used
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_length:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



**6. Training the Language Model**

Now that we’ve built the full model, it’s time to train it and see if it can generate text.
This stage ties everything together — data loading, forward and backward passes, optimization, and evaluation.


In [46]:
# Train and Evaluate the Language Model
#
# Goal:
# Train the Transformer model to predict the next token and periodically evaluate its performance
#
# 1)Initialize model and optimizer
#     - Move the model to the correct device (CPU or GPU).
#     - Use AdamW optimizer for stable training.
#
# 2)Training Loop
#     - For a fixed number of iterations (`max_iters`):
#         a) Evaluate losses on both training and validation sets periodically.
#             • Set model to eval mode (`model.eval()`) to disable dropout etc.
#             • Keep `torch.no_grad()` active during evaluation to save memory and computation.
#             • Compute average loss over multiple `eval_iters` batches.
#             • Print progress to track learning.
#
#         b) Switch back to train mode (`model.train()`).
#             • Sample a training batch from `data_loader('train')`.
#             • Compute logits and loss.
#             • Zero gradients, backpropagate, and step the optimizer.
#
#     - Training is complete once `max_iters` is reached.


model = ...

# create a PyTorch optimizer
optimizer = ...

for iter in range(max_iters):

    # --- add code for evaluating loss on train and val sets ---
    ...

    # --- add code for training step ---
    ...


**7. Text Generation**

After training, we can finally see what our model has learned by generating text autoregressively.
We start with a simple context (often just a single zero token) and let the model predict one token at a time, each new token is fed back as input for the next prediction.

In [ ]:
# Generate text (complete the above cells for this code to run)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=50)[0].tolist()))

Note: This notebook has been created with the help of ChatGPT-5, 15.10.2025; Explanations were initially generated and afterwards edited.